In [1]:
import pandas as pd
import sklearn_crfsuite
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from seqeval.scheme import IOB2

In [2]:
df = pd.read_parquet("../data/df_cleaned.parquet")
df

,idx,tokens,tags_skill,tags_knowledge,source,split,num_skills,num_knowledge,has_overlap,tokens_length
0,1,"[Senior, QA, Engineer, (, m/f/d, ), <ORGANIZAT...","[O, O, O, O, O, O, O]","[O, O, O, O, O, O, O]",tech,train,0,0,False,7
1,1,"[<ADDRESS>, <ADDRESS>, <ADDRESS>, <ADDRESS>, <...","[O, O, O, O, O]","[O, O, O, O, O]",tech,train,0,0,False,5
2,1,"[Date, posted:, 2021-07-14]","[O, O, O]","[O, O, O]",tech,train,0,0,False,3
3,1,"[Likes:, 0, Dislikes:, 0, Love:, 0]","[O, O, O, O, O, O]","[O, O, O, O, O, O]",tech,train,0,0,False,6
4,1,"[Job, description:]","[O, O]","[O, O]",tech,train,0,0,False,2
...,...,...,...,...,...,...,...,...,...,...
11538,65,"[You, can, read, about, the, recruitment, proc...","[O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O]",house,test,0,0,False,10
11539,65,"[Assessment:, The, selected, applications, wil...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]",house,test,0,0,False,14
11540,65,"[242, of, 13, March, 2012, on, the, Appointmen...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",house,test,0,0,False,20
11541,65,"[The, Assessment, Committee, makes, a, non-pri...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",house,test,0,0,False,31


### Functions to Extract Features from the Data

In [3]:
def word2features(sent, i):     # Feature extraction function (based of the sklearn_crfsuite documentation)
    word = sent[i]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[:3]': word[:3],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'word.contains_punct': any(c in '.-/+' for c in word),  # To capture C++, CI/CD, etc.
        'word.is_placeholder': word.startswith('<') and word.endswith('>') # Catch words in <>
    }

    if i > 0:
        word1 = sent[i-1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
        })

    else:
        features['BOS'] = True

    if i < len(sent)-1:
        word1 = sent[i+1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
        })
    else:
        features['EOS'] = True

    if i < len(sent)-2:
        word2 = sent[i+2]
        features.update({
            '+2:word.lower()': word2.lower(),
            '+2:word.istitle()': word2.istitle(),
            '+2:word.isupper()': word2.isupper(),
        })
        
    return features


In [4]:
def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


In [5]:
def format_tags(tags, entity_name):     # format tag for parsing entities (for seqeval)
    return [f"{tag}-{entity_name}" if tag != 'O' else 'O' for tag in tags]

### Converting Data for the Model

In [6]:
X_train = [sent2features(s) for s in df[df['split'] == 'train']['tokens']]
X_val = [sent2features(s) for s in df[df['split'] == 'validation']['tokens']]
X_test = [sent2features(s) for s in df[df['split'] == 'test']['tokens']]

y_train_skill = [format_tags(tags, "SKILL") for tags in df[df["split"] == 'train']['tags_skill']]
y_val_skill = [format_tags(tags, "SKILL") for tags in df[df["split"] == 'validation']['tags_skill']]
y_test_skill = [format_tags(tags, "SKILL") for tags in df[df["split"] == 'test']['tags_skill']]

y_train_knowledge = [format_tags(tags, "KNOWLEDGE") for tags in df[df["split"] == 'train']['tags_knowledge']]
y_val_knowledge = [format_tags(tags, "KNOWLEDGE") for tags in df[df["split"] == 'validation']['tags_knowledge']]
y_test_knowledge = [format_tags(tags, "KNOWLEDGE") for tags in df[df["split"] == 'test']['tags_knowledge']]

### Implementing CRF Model

In [7]:
def tune_crf(X_train, y_train, X_val, y_val):
    best_f1 = 0
    best_crf = None
    best_params = {}

    for c1 in [0.01, 0.1, 1.0]:
        for c2 in [0.01, 0.1, 1.0]:
                crf = sklearn_crfsuite.CRF(
                    algorithm = 'lbfgs',
                    c1 = c1,
                    c2 = c2,
                    max_iterations = 150, # can change?
                    all_possible_transitions = True
                )
                crf.fit(X_train,y_train)

                y_pred_val = crf.predict(X_val)
                f1 = f1_score(y_val, y_pred_val, mode= "strict", scheme=IOB2)

                if f1 > best_f1:
                    best_f1 = f1
                    best_crf = crf
                    best_params = {'c1': c1,
                                   'c2': c2}
                    
    return best_crf, best_params, best_f1

In [8]:
crf_skill = sklearn_crfsuite.CRF(
    algorithm = 'lbfgs',
    c1 = 0.1,
    c2 = 0.1,
    max_iterations = 100, # can change?
    all_possible_transitions = True
)

crf_knowledge = sklearn_crfsuite.CRF(
    algorithm = 'lbfgs',
    c1 = 0.1,
    c2 = 0.1,
    max_iterations = 100, # can change?
    all_possible_transitions = True
)

crf_skill.fit(X_train, y_train_skill)
crf_knowledge.fit(X_train, y_train_knowledge)

y_pred_skill = crf_skill.predict(X_test)
y_pred_knowledge = crf_knowledge.predict(X_test)

print("SKILL EXTRACTION:")
print(classification_report(y_test_skill, y_pred_skill))
print("KNOWLEDGE EXTRACTION:")
print(classification_report(y_test_knowledge, y_pred_knowledge))

SKILL EXTRACTION:
              precision    recall  f1-score   support

       SKILL       0.34      0.17      0.22      1091

   micro avg       0.34      0.17      0.22      1091
   macro avg       0.34      0.17      0.22      1091
weighted avg       0.34      0.17      0.22      1091

KNOWLEDGE EXTRACTION:
              precision    recall  f1-score   support

   KNOWLEDGE       0.58      0.39      0.47      1174

   micro avg       0.58      0.39      0.47      1174
   macro avg       0.58      0.39      0.47      1174
weighted avg       0.58      0.39      0.47      1174



Without Validation:
- The CRF finds only 17% of skills and 39% of knowledge (on recall). Meaning when it does predict a span, it’s right more often (34% and 58%) (on precision). So the model is cautious: it misses most spans rather than guessing wrong.
- The baseline have an F1-score of 0.22 for Skill and 0.47 for Knowledge. Since the model is defined on manual implementation of features, it has limited capability when it comes to extracting abstract and highly variable competencies. However, this is a solid baseline for measuring performance and experimenting with more advanced models like BERT or fine-tune BERT-based model to handle semantic analysis.

In [9]:
print("Tuning Skills CRF")

best_crf_skill, best_params_skill, best_f1_skill = tune_crf(X_train, y_train_skill, X_val, y_val_skill)
print(f"Best Skill Params: {best_params_skill} | Val F1: {best_f1_skill:.4f}")


best_crf_knowledge, best_params_knowledge, best_f1_knowledge = tune_crf(X_train, y_train_knowledge, X_val, y_val_knowledge)
print(f"Best Knowledge Params: {best_params_knowledge} | Val F1: {best_f1_knowledge:.4f}")


Tuning Skills CRF
Best Skill Params: {'c1': 0.01, 'c2': 0.1} | Val F1: 0.2756
Best Knowledge Params: {'c1': 0.1, 'c2': 0.01} | Val F1: 0.4165


In [10]:
y_pred_skill = best_crf_skill.predict(X_test)
y_pred_knowledge = best_crf_knowledge.predict(X_test)

print("Evaluating Skill Extraction Performance (Mode: Strict + IOB2 Theme)")
print(classification_report(y_test_skill, y_pred_skill, mode= "strict", scheme= IOB2))

print("Evaluating Knowledge Extraction Performance (Mode: Strict + IOB2 Theme)")
print(classification_report(y_test_knowledge, y_pred_knowledge, mode= "strict", scheme= IOB2))

Evaluating Skill Extraction Performance (Mode: Strict + IOB2 Theme)
              precision    recall  f1-score   support

       SKILL       0.33      0.18      0.23      1091

   micro avg       0.33      0.18      0.23      1091
   macro avg       0.33      0.18      0.23      1091
weighted avg       0.33      0.18      0.23      1091

Evaluating Knowledge Extraction Performance (Mode: Strict + IOB2 Theme)
              precision    recall  f1-score   support

   KNOWLEDGE       0.55      0.41      0.47      1174

   micro avg       0.55      0.41      0.47      1174
   macro avg       0.55      0.41      0.47      1174
weighted avg       0.55      0.41      0.47      1174



From the tuned CRF, the improvement is minimal because the CRF fundamentally cannot understand the underlying meaning of the text. It can only look for the exact grammatical and formatting patterns that was hardcoded.

The tuned CRF baseline establishes our floor for model comparison. The results reveal two core challenges in this dataset: 
- (1) The model is highly cautious, achieving decent precision but very low recall—meaning it misses the vast majority of spans rather than guessing incorrectly. 
- (2) Skill extraction is significantly harder than Knowledge extraction (mirroring the EDA findings, as Skill spans are typically multi-word phrases compared to single-word Knowledge entities like "Python"). 
- Hyperparameters were selected strictly on the validation set, and all scores were calculated using strict IOB2 span evaluation to ensure a fair, rigorous comparison with future BERT models.

In [12]:
import os

results = []

for entity, y_true, y_pred in [("SKILL", y_test_skill, y_pred_skill), ("KNOWLEDGE", y_test_knowledge, y_pred_knowledge)]:
    results.append({
        'Model': 'CRF Baseline (Tuned)',
        "Entity": entity,
        "Precision": round(precision_score(y_true, y_pred, mode="strict", scheme=IOB2), 4),
        "Recall": round(recall_score(y_true, y_pred, mode="strict", scheme=IOB2), 4),
        "F1_Score": round(f1_score(y_true, y_pred, mode="strict", scheme=IOB2), 4)
    })

result_df = pd.DataFrame(results)

os.makedirs('../results', exist_ok = True)
result_df.to_csv('../results/crf_baseline_results.csv',index = False)